# BÀI TẬP THỰC HÀNH HỒI QUY TUYẾN TÍNH (BÀI 1)
---
### Bài tập 1: Lọc ra nhóm căn hộ lớn (> 100 m²)

In [ ]:
import pandas as pd

# 1. Đọc dữ liệu
df = pd.read_csv("../data/gia_nha.csv")

# 2. Lọc các căn hộ có diện tích > 100 m2
nhom_lon = df[df["dien_tich"] > 100]
so_luong = len(nhom_lon)
gia_trung_binh = nhom_lon["gia"].mean()

# 3. In kết quả
print(f"Số căn hộ trên 100 mét vuông là: {so_luong} căn")
print(f"Giá trung bình của riêng nhóm căn đó là: {gia_trung_binh:.3f} tỷ đồng")

---
### Bài tập 2: Vẽ biểu đồ phân tán theo số phòng

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.scatter(df["so_phong"], df["gia"], color="royalblue", edgecolors="black", s=60, alpha=0.8)
plt.title("Biểu đồ phân tán giữa số phòng và giá nhà", fontsize=13)
plt.xlabel("Số phòng ngủ (phòng)", fontsize=11)
plt.ylabel("Giá bán (tỷ đồng)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)

# Lưu hình trước khi show
plt.savefig("bai2.png", dpi=150, bbox_inches="tight")
plt.show()

print("Nhận xét: Nhìn chung số phòng ngủ càng nhiều thì giá nhà càng có xu hướng tăng.")
print("Tuy nhiên ở cùng một số phòng (ví dụ 2 hoặc 3 phòng), mức giá dao động khá lớn do phụ thuộc vào diện tích.")

---
### Bài tập 3: Đổi biến đầu vào sang tuổi nhà

In [ ]:
import numpy as np

x = df["tuoi_nha"].to_numpy()
y = df["gia"].to_numpy()

x_tb = x.mean()
y_tb = y.mean()

# Áp dụng công thức bình phương tối thiểu ở mục 5
tu_so = ((x - x_tb) * (y - y_tb)).sum()
mau_so = ((x - x_tb) ** 2).sum()

w = tu_so / mau_so
b = y_tb - w * x_tb

print(f"Hệ số góc  w = {w:.6f}")
print(f"Hệ số chặn b = {b:.6f}")
print()
print(f"Mô hình: gia = {w:.6f} * tuoi_nha + {b:.6f}")
print()
print("Nhận xét về dấu của w:")
print("Hệ số góc w mang dấu âm (-0.037858), thể hiện mối quan hệ nghịch biến:")
print("Tuổi nhà càng tăng (nhà càng cũ, hao mòn theo thời gian) thì giá trị căn hộ càng giảm.")
print(f"Cụ thể, mỗi năm tuổi nhà tăng thêm làm giá nhà giảm trung bình khoảng {abs(w)*1000:.2f} triệu đồng.")

---
### Bài tập 4: Thêm số phòng vào mô hình (Mô hình 2 biến)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# 1. Đầu vào gồm 2 đặc trưng: dien_tich và so_phong
X = df[["dien_tich", "so_phong"]]
y = df["gia"]

# 2. Chia tập học và tập kiểm tra (giữ nguyên test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Huấn luyện mô hình
mo_hinh = LinearRegression()
mo_hinh.fit(X_train, y_train)

# 4. Đánh giá R2 trên tập kiểm tra
y_pred = mo_hinh.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"R2 trên tập kiểm tra (mô hình 2 biến): {r2:.4f}")
print("So sánh với mô hình 1 biến (chỉ dùng diện tích có R2 = 0.9622):")
print(f"R2 tăng từ 0.9622 lên {r2:.4f} (tăng thêm {r2 - 0.9622:.4f}).")
print()
print("Nhận xét: Việc thêm cột so_phong có cải thiện độ chính xác của mô hình,")
print("tuy nhiên mức tăng không quá lớn vì dien_tich và so_phong có sự tương quan cao")
print("(căn có diện tích lớn thường đã có nhiều phòng ngủ sẵn).")

---
### Bài tập 5: Thử hai tốc độ học khác (Gradient Descent)

In [ ]:
# Chuẩn hóa dữ liệu diện tích quanh mốc 0
x_goc = df["dien_tich"].to_numpy()
y_goc = df["gia"].to_numpy()

x_tb = x_goc.mean()
x_do_lech = x_goc.std()
x_chuan_hoa = (x_goc - x_tb) / x_do_lech
n = len(x_chuan_hoa)
so_vong = 200

ket_qua_mse = {}

for toc_do_hoc in [0.001, 1.02]:
    w, b = 0.0, 0.0
    for vong in range(1, so_vong + 1):
        y_du_doan = w * x_chuan_hoa + b
        chenh_lech = y_du_doan - y_goc
        grad_w = (2 / n) * (chenh_lech * x_chuan_hoa).sum()
        grad_b = (2 / n) * chenh_lech.sum()
        w -= toc_do_hoc * grad_w
        b -= toc_do_hoc * grad_b
    
    mse_200 = (((w * x_chuan_hoa + b) - y_goc) ** 2).mean()
    ket_qua_mse[toc_do_hoc] = mse_200
    print(f"Tốc độ học = {toc_do_hoc:5.3f} -> MSE ở vòng 200 là: {mse_200:.4f}")

print()
print("Giải thích sự khác biệt giữa hai con số:")
print("1. Với tốc độ học = 0.001 (quá nhỏ): Mỗi bước cập nhật quá ngắn, sau 200 vòng thuật toán")
print("   chưa kịp xuống tới đáy lòng chảo nên MSE vẫn còn lớn (khoảng 20.2175, chưa hội tụ).")
print("2. Với tốc độ học = 1.02 (quá lớn): Bước đi dài vượt quá đáy, nhảy sang sườn bên kia")
print("   và ngày càng bị văng ra xa (phân kỳ), khiến sai số bùng nổ (MSE lên tới hàng trăm triệu).")

---
### Bài tập 6: Viết hàm dự đoán có cảnh báo

In [ ]:
# Sử dụng 2 hệ số từ mô hình 1 biến ở mục 5
w = 0.078367
b = 0.401752

def du_doan_gia(dien_tich):
    """
    Dự đoán giá căn hộ từ diện tích.
    Có cảnh báo nếu diện tích nằm ngoài khoảng quan sát trong dữ liệu (35.5 - 117.5 m2).
    """
    if dien_tich < 35.5 or dien_tich > 117.5:
        print(f"CẢNH BÁO: Diện tích {dien_tich} m2 nằm ngoài khoảng dữ liệu huấn luyện (35.5 - 117.5 m2)!")
        print("          Kết quả dự đoán có tính chất ngoại suy và có thể không đáng tin cậy.")
    gia_du_doan = w * dien_tich + b
    return gia_du_doan

# Chạy thử với 3 mốc: 60, 80 và 200 m2
for dt in [60, 80, 200]:
    print(f"--- Dự đoán căn hộ {dt} m2 ---")
    gia = du_doan_gia(dt)
    print(f"=> Giá dự đoán: {gia:.3f} tỷ đồng\n")